# Load Dataset and Evaluate Persistent Excitation

This notebook:
1. Loads a generated random-walk dataset from disk.
2. Selects an arbitrary trajectory example.
3. Plots the selected trajectory (states + input).
4. Evaluates persistent excitation (PE) for the selected input and summarizes PE across all trajectories.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path("../src").resolve()))

from io_utils import load_dataset_npz
from pe import persistent_excitation_check, readiness_checks
from visualization import plot_states_and_input

In [ ]:
# Point to your dataset file
dataset_path = Path("../data/full_example/random_walk_dataset.npz")

if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at {dataset_path}. Run notebooks/full_example.ipynb first."
    )

data = load_dataset_npz(dataset_path)
states = data["states"]      # (N, T, 4)
inputs = data["inputs"]      # (N, T)
t = data["t"]                # (T,)
initial_state = data["initial_state"]

state_labels = ["x (m)", "x_dot (m/s)", "theta (rad)", "theta_dot (rad/s)"]

print("Loaded:", dataset_path)
print("states shape:", states.shape)
print("inputs shape:", inputs.shape)
print("time shape:", t.shape)
print("stored initial state:", initial_state)
print("state order:", state_labels)

In [ ]:
# Choose an arbitrary example trajectory index
rng = np.random.default_rng(2026)
example_idx = int(rng.integers(low=0, high=states.shape[0]))

print("Selected example index:", example_idx)

y_example = states[example_idx].T   # shape -> (4, T)
u_example = inputs[example_idx]      # shape -> (T,)

print("\nState summary for selected trajectory:")
for i, label in enumerate(state_labels):
    series = y_example[i]
    print(f"  {label}: min={series.min():.6f}, max={series.max():.6f}")

fig, _ = plot_states_and_input(
    t,
    y_example,
    u_example,
    title=f"Trajectory Example #{example_idx}",
    theta_in_degrees=True,
 )
plt.show()

In [ ]:
# Evaluate PE for the selected example and summarize PE across all trajectories
pe_order = 20

pe_example = persistent_excitation_check(u_example, order=pe_order)
print("Example trajectory PE check")
print("  order:", pe_example.order)
print("  rank:", pe_example.rank)
print("  min_dim:", pe_example.min_dim)
print("  pass:", pe_example.is_persistently_exciting)
print("  condition number:", pe_example.condition_number)

pe_results = [persistent_excitation_check(inputs[i], order=pe_order) for i in range(inputs.shape[0])]
num_pass = sum(int(r.is_persistently_exciting) for r in pe_results)

print("\nDataset-wide PE summary")
print("  trajectories:", inputs.shape[0])
print("  PE passes:", num_pass)
print("  pass ratio:", num_pass / inputs.shape[0])

readiness = readiness_checks(states, inputs, t)
print("\nReadiness checks:")
for k, v in readiness.items():
    print(f"  {k}: {v}")